# Preprocessing for Power BI

In [15]:
import pandas as pd

## Import Patterns

In [16]:
# Read data and fill NA values with False before converting to boolean
allpatterns = pd.read_csv('complete_patterns.csv', header=0, low_memory=False)
if 'free_status' in allpatterns.columns:
    allpatterns['free_status'] = allpatterns['free_status'].fillna(False).astype(bool)
if 'is_clothing' in allpatterns.columns:
    allpatterns['is_clothing'] = allpatterns['is_clothing'].fillna(False).astype(bool)

allpatterns.head()

,pattern_id,free_status,projects_count,yarn_weight,craft,attributes,is_clothing,supercategory,category,subcategory,...,pattern_author,author_id,price,currency,queued_projects_count,rating_average,rating_count,yardage,yarn_ids,pattern_source_type_names
0,7516797,False,0,Sport,Knitting,"['female', 'adult', '3-4-sleeve', 'stranded', ...",True,Clothing,Sweater,Pullover,...,Annalisa Filippi,119196,4.00,EUR,9,0.000000,NaN,910.0,[118411],['Ravelry Store']
1,7517071,False,0,Sport,Knitting,"['unisex', 'adult', 'phototutorial', 'written-...",True,Clothing,Sweater,Pullover,...,Smith Olga,156643,13.31,EUR,4,0.000000,NaN,175.0,"[24537, 52662, 103836, 158857, 165070]","['Website', 'Ravelry Store']"
2,7516775,True,3,Sport,Crochet,"['female', 'unisex', 'adult', 'lace', 'reversi...",True,Clothing,Tops,Tee,...,Kaja Drnovšek,160454,NaN,USD,3,0.000000,NaN,500.0,[219391],['Website']
3,7446187,False,1182,Sport,Knitting,"['stripes', 'textured', 'circular-yoke', 'crew...",True,Clothing,Sweater,Pullover,...,Andrea Mowry,78156,9.00,USD,993,4.794776,268.0,1100.0,"[102622, 226436]",['Ravelry Store']
4,7516957,False,9,Sport,Knitting,"['female', 'baby', 'newborn-size', 'toddler', ...",True,Clothing,Dress,NaN,...,Taiga Hilliard,42962,6.50,USD,3,0.000000,NaN,300.0,[74654],['Ravelry Store']


In [17]:
# Drop duplicate pattern_id rows, keeping the first occurrence
allpatterns = allpatterns.drop_duplicates(subset='pattern_id', keep='first')


## Load and Merge Yarns

In [18]:
most_yarns = pd.read_csv('yarns.csv', header=0, low_memory=False)
missing_yarns = pd.read_csv('missing_yarns.csv', header=0, low_memory=False)

yarns = pd.concat([most_yarns, missing_yarns], ignore_index=True)

## Clean

In [19]:
# Drop rating_average and rating_count
allpatterns = allpatterns.drop(columns=['rating_average', 'rating_count'], errors='ignore')

# Convert created_at to datetime
allpatterns['created_at'] = pd.to_datetime(allpatterns['created_at'], errors='coerce', utc=True)

# Convert has_uk_terminology and has_us_terminology to boolean
allpatterns['has_uk_terminology'] = allpatterns['has_uk_terminology'].fillna(False).astype(bool)
allpatterns['has_us_terminology'] = allpatterns['has_us_terminology'].fillna(False).astype(bool)

In [20]:
# Drop every pattern published after February 28, 2026
allpatterns = allpatterns.sort_values('created_at', ascending=False).reset_index(drop=True)
allpatterns = allpatterns[allpatterns['created_at'].dt.date <= pd.to_datetime('2026-02-28').date()].reset_index(drop=True)

In [21]:
# Create meteorological `season` column
def get_season(date):
	if pd.isna(date):
		return 'Unknown'
	month = date.month
	if month in [12, 1, 2]:
		return 'Winter'
	elif month in [3, 4, 5]:
		return 'Spring'
	elif month in [6, 7, 8]:
		return 'Summer'
	else:
		return 'Fall'
	
allpatterns['season'] = allpatterns['created_at'].apply(get_season)

In [22]:
# Convert to dummy variables
# Attributes
attributes_expanded = allpatterns['attributes'].str.split(',', expand=False).explode().str.strip("[]' ")
attributes_dummies = pd.get_dummies(attributes_expanded, prefix='attributes')
attributes_dummies = attributes_dummies.groupby(attributes_dummies.index).max()

# Languages
languages_expanded = allpatterns['languages'].str.split(',', expand=False).explode().str.strip("[]' ")
languages_dummies = pd.get_dummies(languages_expanded, prefix='languages')
languages_dummies = languages_dummies.groupby(languages_dummies.index).max()

# Pattern_source_type_names
pattern_source_type_names_expanded = allpatterns['pattern_source_type_names'].str.split(',', expand=False).explode().str.strip("[]' ")
pattern_source_type_names_dummies = pd.get_dummies(pattern_source_type_names_expanded, prefix='pattern_source_type_names')
pattern_source_type_names_dummies = pattern_source_type_names_dummies.groupby(pattern_source_type_names_dummies.index).max()

# Join and clean up
allpatterns = allpatterns.join(attributes_dummies)
allpatterns = allpatterns.join(languages_dummies)
allpatterns = allpatterns.join(pattern_source_type_names_dummies)
allpatterns = allpatterns.drop(columns=['attributes', 'languages', 'pattern_source_type_names'], errors='ignore')
allpatterns.head()

,pattern_id,free_status,projects_count,yarn_weight,craft,is_clothing,supercategory,category,subcategory,babycategory,...,languages_Welsh,pattern_source_type_names_,pattern_source_type_names_App,pattern_source_type_names_Book,pattern_source_type_names_Magazine,pattern_source_type_names_Pamphlet,pattern_source_type_names_Ravelry Store,pattern_source_type_names_Website,pattern_source_type_names_Webzine,pattern_source_type_names_eBook
0,7508203,False,0,DK,Knitting,True,Clothing,Sweater,Cardigan,NaN,...,False,False,False,False,False,False,True,False,False,False
1,7508202,True,0,NaN,Crochet,False,Toys and Hobbies,Doll Clothes,Child Doll,NaN,...,False,False,False,False,False,False,True,False,False,True
2,7508201,True,0,Light Fingering,Crochet,False,Toys and Hobbies,Softies,Animal,NaN,...,False,False,False,False,False,False,False,True,False,False
3,7508200,True,2,Worsted,Crochet,False,Home,Decorative,Other,Toys and Hobbies,...,False,False,False,False,False,False,True,False,False,False
4,7508199,True,2,Bulky,Crochet,True,Accessories,Hat,"Beanie, Toque",NaN,...,False,False,False,False,False,False,False,True,False,False


## Engineer columns

In [23]:
allpatterns.head()

,pattern_id,free_status,projects_count,yarn_weight,craft,is_clothing,supercategory,category,subcategory,babycategory,...,languages_Welsh,pattern_source_type_names_,pattern_source_type_names_App,pattern_source_type_names_Book,pattern_source_type_names_Magazine,pattern_source_type_names_Pamphlet,pattern_source_type_names_Ravelry Store,pattern_source_type_names_Website,pattern_source_type_names_Webzine,pattern_source_type_names_eBook
0,7508203,False,0,DK,Knitting,True,Clothing,Sweater,Cardigan,NaN,...,False,False,False,False,False,False,True,False,False,False
1,7508202,True,0,NaN,Crochet,False,Toys and Hobbies,Doll Clothes,Child Doll,NaN,...,False,False,False,False,False,False,True,False,False,True
2,7508201,True,0,Light Fingering,Crochet,False,Toys and Hobbies,Softies,Animal,NaN,...,False,False,False,False,False,False,False,True,False,False
3,7508200,True,2,Worsted,Crochet,False,Home,Decorative,Other,Toys and Hobbies,...,False,False,False,False,False,False,True,False,False,False
4,7508199,True,2,Bulky,Crochet,True,Accessories,Hat,"Beanie, Toque",NaN,...,False,False,False,False,False,False,False,True,False,False


In [24]:
sorted_patterns = allpatterns.sort_values('author_id', ascending=True).reset_index(drop=True)

# Create column: previously_published_patterns (int): sum of all previously published patterns by the same author
for author in sorted_patterns['author_id'].unique():
	author_patterns = sorted_patterns[sorted_patterns['author_id'] == author].sort_values('created_at', ascending=True).reset_index()

	already_published = 0
	free_pattern = False
	
	for idx, row in author_patterns.iterrows():
		pattern_id = row['pattern_id']
		original_idx = sorted_patterns[sorted_patterns['pattern_id'] == pattern_id].index[0]
		
		sorted_patterns.loc[original_idx, 'previously_published_patterns'] = int(already_published)
		already_published += 1

		if not free_pattern and row['free_status']:
			free_pattern = True
		
		sorted_patterns.loc[original_idx, 'free_patterns'] = free_pattern

In [25]:
# Verify the previously_published_patterns column
author_patterns = sorted_patterns[sorted_patterns['author_id'] == 803].sort_values('created_at', ascending=True).reset_index()
author_patterns[['created_at', 'free_status', 'author_id', 'previously_published_patterns']].head()

,created_at,free_status,author_id,previously_published_patterns
0,2007-05-06 02:21:06+00:00,True,803,0.0
1,2007-05-14 04:24:34+00:00,True,803,1.0
2,2007-06-10 10:25:16+00:00,True,803,2.0
3,2007-06-10 10:26:06+00:00,True,803,3.0
4,2007-06-10 10:27:14+00:00,True,803,4.0


In [26]:
# Remove all records where free_status is True
sorted_patterns = sorted_patterns[~sorted_patterns['free_status']].reset_index(drop=True)

In [27]:
# Drop free_status
sorted_patterns = sorted_patterns.drop(columns=['free_status'], errors='ignore')

In [28]:
# Create projects_per_day - # of projects divided by days_since_publication
# Pattern data was collected April 1-3, 2026
sorted_patterns['days_since_publication'] = (pd.Timestamp('2026-04-03', tz='UTC') - sorted_patterns['created_at']).dt.days
sorted_patterns['projects_per_day'] = sorted_patterns['projects_count'] / sorted_patterns['days_since_publication'].replace(0, 1)  # Avoid division by zero
sorted_patterns = sorted_patterns.drop(columns=[['created_at', 'projects_count']], errors='ignore')

## Join yarn attributes

In [ ]:
# Drop duplicate IDs
yarns = yarns.drop_duplicates(subset=['id']).reset_index(drop=True)

In [ ]:
# Remove brackets from yarn_ids
sorted_patterns['yarn_ids'] = sorted_patterns['yarn_ids'].str.strip("[]' ")

In [ ]:
# Index yarns by id for faster lookup
yarn_dict = yarns.set_index('id').to_dict(orient='index')
print(yarn_dict[3941])

{'animal_fiber': False, 'synthetic_fiber': False, 'vegetable_fiber': True, 'yarn_fiber_names': "['Cotton']", 'yarn_weight': 'Thread'}


In [ ]:
# For each pattern, join yarn attributes by yarn ID
for pattern in sorted_patterns['pattern_id'].unique():
	if pd.notna(sorted_patterns.loc[sorted_patterns['pattern_id'] == pattern, 'yarn_ids'].iloc[0]):
		# Split yarn_ids into array (if array is not empty)
		yarn_ids_str = sorted_patterns.loc[sorted_patterns['pattern_id'] == pattern, 'yarn_ids'].iloc[0]
		if yarn_ids_str != '':
			yarn_id_array = [int(yarn_id.strip()) for yarn_id in yarn_ids_str.split(',')]
		else:
			yarn_id_array = []
		
		# Get yarn attributes for the current pattern
		yarn_attributes = [yarn_dict[yarn_id] for yarn_id in yarn_id_array if yarn_id in yarn_dict]

		# Set yarn attribute variables
		animal_fiber = any(yarn.get('animal_fiber') == True for yarn in yarn_attributes)
		synthetic_fiber = any(yarn.get('synthetic_fiber') == True for yarn in yarn_attributes)
		vegetable_fiber = any(yarn.get('vegetable_fiber') == True for yarn in yarn_attributes)
		yarn_fiber_names = ', '.join(set([yarn.get('yarn_fiber_names', '') for yarn in yarn_attributes if yarn.get('yarn_fiber_names')]))
		
		# Get yarn_weight - prefer non-Unknown values
		yarn_weights = [yarn.get('yarn_weight') for yarn in yarn_attributes if yarn.get('yarn_weight') and yarn.get('yarn_weight') != 'Unknown']
		yarn_weight = yarn_weights[0] if yarn_weights else sorted_patterns.loc[sorted_patterns['pattern_id'] == pattern, 'yarn_weight'].iloc[0]
		
		# Update the pattern record
		original_idx = sorted_patterns[sorted_patterns['pattern_id'] == pattern].index[0]
		sorted_patterns.loc[original_idx, 'animal_fiber'] = animal_fiber
		sorted_patterns.loc[original_idx, 'synthetic_fiber'] = synthetic_fiber
		sorted_patterns.loc[original_idx, 'vegetable_fiber'] = vegetable_fiber
		sorted_patterns.loc[original_idx, 'yarn_fiber_names'] = yarn_fiber_names
		sorted_patterns.loc[original_idx, 'yarn_weight'] = yarn_weight

In [ ]:
sorted_patterns[['yarn_weight', 'yarn_fiber_names', 'animal_fiber', 'synthetic_fiber', 'vegetable_fiber']].head()

,yarn_weight,yarn_fiber_names,animal_fiber,synthetic_fiber,vegetable_fiber
0,Aran,['Silk'],True,False,False
1,Fingering,"['Nylon', 'Cashmere', 'Merino']",True,True,False
2,Lace,['Merino'],True,False,False
3,Worsted,,False,False,False
4,DK,"['Silk', 'Alpaca']",True,False,False


In [ ]:
sorted_patterns.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 801689 entries, 0 to 801688
Data columns (total 333 columns):
 #    Column                                     Dtype              
---   ------                                     -----              
 0    pattern_id                                 int64              
 1    projects_count                             int64              
 2    yarn_weight                                str                
 3    craft                                      str                
 4    is_clothing                                bool               
 5    supercategory                              str                
 6    category                                   str                
 7    subcategory                                str                
 8    babycategory                               str                
 9    created_at                                 datetime64[us, UTC]
 10   favorites_count                            int64              
 1

In [ ]:
# Convert yarn_fiber_names to dummy variables
# Handle empty/null values
yarn_fiber_expanded = sorted_patterns['yarn_fiber_names'].fillna('').str.split(',', expand=False).explode().str.strip("[]' ")
yarn_fiber_expanded = yarn_fiber_expanded[yarn_fiber_expanded != '']  # Remove empty strings
yarn_fiber_dummies = pd.get_dummies(yarn_fiber_expanded, prefix='yarn_fiber')
yarn_fiber_dummies = yarn_fiber_dummies.groupby(yarn_fiber_dummies.index).max()

sorted_patterns = sorted_patterns.join(yarn_fiber_dummies)
sorted_patterns.drop(columns=['yarn_fiber_names'], inplace=True)

sorted_patterns.head()

,pattern_id,projects_count,yarn_weight,craft,is_clothing,supercategory,category,subcategory,babycategory,created_at,...,yarn_fiber_Other,yarn_fiber_Plant fiber,yarn_fiber_Polyester,yarn_fiber_Qiviut,yarn_fiber_Rayon,yarn_fiber_Silk,yarn_fiber_Soy,yarn_fiber_Tencel,yarn_fiber_Wool,yarn_fiber_Yak
0,81231,65,Aran,Knitting,True,Accessories,Neck / Torso,Scarf,NaN,2008-08-21 21:46:00+00:00,...,False,False,False,False,False,True,False,False,False,False
1,138972,235,Fingering,Knitting,True,Accessories,Hat,"Beret, Tam",NaN,2009-09-03 17:04:15+00:00,...,False,False,False,False,False,False,False,False,False,False
2,318176,60,Lace,Knitting,True,Accessories,Neck / Torso,Shawl / Wrap,NaN,2012-05-05 01:31:17+00:00,...,False,False,False,False,False,False,False,False,False,False
3,318174,6,Worsted,Knitting,True,Accessories,Neck / Torso,Cowl,NaN,2012-05-05 01:22:45+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,318170,14,DK,Knitting,True,Accessories,Neck / Torso,Shawl / Wrap,NaN,2012-05-05 01:15:42+00:00,...,False,False,False,False,False,True,False,False,False,False


In [ ]:
# Save to pickle
sorted_patterns.to_pickle('checkpoint.pkl')